# Data exploration & quality

Checks on the extract itself, no model needed (CPU only). Each customer is either
**flagged** (at least one flagged transfer) or **normal**.

1. When does each customer's activity end? If flagged customers stop earlier, the date of
   their transfers alone gives them away, and any model can use that as a shortcut.
2. Share of each group still active, month by month.
3. How well time and activity alone predict the flag. Notebook 07's scores must beat this.

In [ ]:
# ======================================================================
# EDIT HERE
# ======================================================================
WINDOW = 32   # history window used in notebook 07
# ======================================================================

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.experiment import shared_data_dir

COLORS = {0: "#2a78d6", 1: "#eb6834"}   # normal, flagged
NAMES = {0: "normal", 1: "flagged"}

d = shared_data_dir()
schema = json.loads((d / "tokenizer" / "schema.json").read_text())
cust, time = schema["customer_id"], schema["datetime"]
df = pd.concat([pd.read_parquet(d / "temporal_split" / f"{s}.parquet", columns=[cust, time, "_target"])
                for s in ("train", "val", "test")], ignore_index=True)
df[time] = pd.to_datetime(df[time])
df = df.sort_values([cust, time])
end = df[time].max()

# One row per customer
g = df.groupby(cust)
c = g.agg(flagged=("_target", "max"), first=(time, "min"), last=(time, "max"), n_transfers=(time, "size"))
start = df[g.cumcount(ascending=False) == WINDOW - 1].set_index(cust)[time]
c["window_start"] = start.reindex(c.index).fillna(c["first"])
c["days_since_last"] = (end - c["last"]).dt.days
c["days_active"] = (c["last"] - c["first"]).dt.days
c["window_days"] = (c["last"] - c["window_start"]).dt.days
print(f"{len(c):,} customers, {c['flagged'].sum():,} flagged ({c['flagged'].mean():.2%}); "
      f"extract {df[time].min().date()} to {end.date()}")

## 1. When does each customer's activity end?

Share of customers (y) whose last transfer is at most x days before the end of the extract.
Two curves on top of each other = no time difference.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
for f in (0, 1):
    x = np.sort(c.loc[c["flagged"] == f, "days_since_last"].values)
    ax.plot(x, np.arange(1, len(x) + 1) / len(x), lw=2, color=COLORS[f], label=NAMES[f])
ax.set_xlabel("days between last transfer and end of extract")
ax.set_ylabel("share of customers")
ax.grid(alpha=0.3)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(c.groupby("flagged")["days_since_last"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).unstack()
      .rename(index=NAMES).to_string())

## 2. Share of each group active, month by month

A customer counts as active in a month if they made at least one transfer in it.

In [ ]:
month = df[time].dt.to_period("M")
flag = df[cust].map(c["flagged"])
active = df.groupby([month, flag])[cust].nunique().unstack() / c["flagged"].value_counts()

fig, ax = plt.subplots(figsize=(9, 3.8))
for f in (0, 1):
    ax.plot(active.index.to_timestamp(), active[f], lw=2, color=COLORS[f], label=NAMES[f])
ax.set_ylabel("share of group active")
ax.set_ylim(0, None)
ax.grid(alpha=0.3)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 3. Can time or activity alone predict the flag?

Each value on its own, used as a score. ROC-AUC near 0.5 and AP near the share of flagged
customers = no shortcut. Direction does not matter (higher or lower both count).

In [ ]:
y = c["flagged"].astype(int)
rows = []
for col in ["days_since_last", "days_active", "n_transfers", "window_days"]:
    auc = roc_auc_score(y, c[col])
    ap = max(average_precision_score(y, c[col]), average_precision_score(y, -c[col]))
    rows.append({"value": col, "AUC": max(auc, 1 - auc), "AP": ap})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"AP by chance: {y.mean():.3f}")

## 4. Fixes: what would each one cost?

If part 3 shows a time shortcut, the two groups' windows must cover the same periods.
Four options, with how many customers each keeps:

- **4a. One common date D for everyone.** Use only transfers before D, keep customers
  active in the `ACTIVE_DAYS` before D. Simple, but flagged customers who stopped before D are lost.
- **4b. Matched end dates.** Each normal customer gets a date drawn from the flagged
  customers' end dates; their window ends at their own transfer closest to it. Keeps every
  flagged customer.
- **4c. Score within periods.** Keep everything, but compute ROC-AUC inside groups of
  customers whose window ends in the same quarter, then average. Time cannot help inside a group.
- **4d. Split by customer and by time.** Train = customers whose last transfer is before T,
  test = other customers whose last transfer is after T. Works only if enough flagged
  customers are still active after T.

*Why not train only on customers of the same period?* If the share of flagged customers differs
between periods, the model still learns "this period → flagged", even with the folds
restricted. What removes the shortcut is the same share of flagged customers in every period
(4b) or comparing only customers of the same period (4c).

In [ ]:
ACTIVE_DAYS = 90   # 4a: a customer counts if they made a transfer in the ACTIVE_DAYS before D

# 4a. Customers kept for each candidate date D (first day of each month)
dates = pd.date_range(df[time].min() + pd.Timedelta(days=ACTIVE_DAYS), end, freq="MS")
kept = []
for D in dates:
    before = df[df[time] < D]
    recent = before.loc[before[time] >= D - pd.Timedelta(days=ACTIVE_DAYS), cust].unique()
    n_before = before.groupby(cust).size().reindex(recent)
    fl = c.loc[recent, "flagged"]
    kept.append({"D": D, "flagged": int(fl.sum()), "normal": int((fl == 0).sum()),
                 f"flagged with {WINDOW}+ transfers": int(((n_before >= WINDOW) & (fl == 1)).sum())})
kept = pd.DataFrame(kept).set_index("D")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(kept.index, kept["flagged"], lw=2, color=COLORS[1], label="flagged kept")
axes[0].plot(kept.index, kept[f"flagged with {WINDOW}+ transfers"], lw=2, ls="--", color=COLORS[1],
             label=f"of which {WINDOW}+ transfers before D")
axes[0].axhline(c["flagged"].sum(), color="grey", ls=":", lw=1, label="all flagged")
axes[1].plot(kept.index, kept["normal"], lw=2, color=COLORS[0], label="normal kept")
axes[1].axhline((c["flagged"] == 0).sum(), color="grey", ls=":", lw=1, label="all normal")
for ax in axes:
    ax.set_xlabel("common date D")
    ax.set_ylim(0, None)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_title("4a. Flagged customers kept")
axes[1].set_title("4a. Normal customers kept")
plt.tight_layout()
plt.show()
best = kept["flagged"].idxmax()
print(f"Best D: {best.date()}, keeps {kept.loc[best, 'flagged']:,} of {c['flagged'].sum():,} flagged "
      f"and {kept.loc[best, 'normal']:,} of {(c['flagged'] == 0).sum():,} normal customers")

In [ ]:
# 4b. Matched end dates: each normal customer gets an end date drawn from the flagged customers' end dates
rng = np.random.default_rng(0)
normal = c.index[c["flagged"] == 0]
draw = pd.DataFrame({cust: normal, "draw": rng.choice(c.loc[c["flagged"] == 1, "last"].values, size=len(normal))})
# The window ends at the customer's own transfer closest to the drawn date
near = pd.merge_asof(draw.sort_values("draw"), df[df[cust].isin(normal)][[cust, time]].sort_values(time),
                     left_on="draw", right_on=time, by=cust, direction="nearest")
cut = c["last"].copy()   # flagged customers keep their own end
cut[near[cust].values] = near[time].values

t = df[df[time] <= df[cust].map(cut)]
m = t.groupby(cust)[time].agg(["max", "size"]).reindex(c.index)
has = m["size"].notna()
print(f"4b keeps {c.loc[has, 'flagged'].sum():,} of {c['flagged'].sum():,} flagged and "
      f"{(has & (c['flagged'] == 0)).sum():,} of {len(normal):,} normal customers "
      f"({(m['size'] >= WINDOW)[normal].mean():.0%} of normal with {WINDOW}+ transfers before the cut)")

# Is time still a shortcut after matching? Should be close to AUC 0.5, AP at chance.
days = (end - m.loc[has, "max"]).dt.days
y4 = c.loc[has, "flagged"].astype(int)
auc = roc_auc_score(y4, days)
ap = max(average_precision_score(y4, days), average_precision_score(y4, -days))
print(f"days_since_last after matching: AUC {max(auc, 1 - auc):.3f}  AP {ap:.3f}  (AP by chance {y4.mean():.3f})")

In [ ]:
# 4c. Score within periods: customers grouped by the quarter their window ends
q = c["last"].dt.to_period("Q")
groups = c.groupby(q)["flagged"].agg(customers="size", flagged="sum")
print(groups.to_string())
usable = groups.index[(groups["flagged"] > 0) & (groups["flagged"] < groups["customers"])]
print(f"\nQuarters with both groups: {len(usable)}, holding {groups.loc[usable, 'flagged'].sum():,} "
      f"of {c['flagged'].sum():,} flagged customers")

if len(usable):
    # Time alone, scored within quarters (weighted by flagged customers): should be close to 0.5
    aucs = [(roc_auc_score(c.loc[q == p, "flagged"], c.loc[q == p, "days_since_last"]), groups.loc[p, "flagged"])
            for p in usable]
    w_auc = sum(a * n for a, n in aucs) / sum(n for _, n in aucs)
    print(f"days_since_last within quarters: AUC {max(w_auc, 1 - w_auc):.3f}")

### 4d. Split by customer and by time: enough flagged customers on each side?

For each cut date T: customers whose last transfer is before T (train) and after T (test).
The dashed lines are the val and test start dates of the time split.

In [ ]:
cuts = pd.date_range(c["last"].min().normalize(), end, freq="MS")[1:]
side = pd.DataFrame({"T": cuts,
                     "train flagged": [int(((c["last"] < T) & (c["flagged"] == 1)).sum()) for T in cuts],
                     "test flagged": [int(((c["last"] >= T) & (c["flagged"] == 1)).sum()) for T in cuts],
                     "test normal": [int(((c["last"] >= T) & (c["flagged"] == 0)).sum()) for T in cuts]}).set_index("T")

split_starts = [pd.read_parquet(d / "temporal_split" / f"{s}.parquet", columns=[time])[time].min()
                for s in ("val", "test")]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(side.index, side["train flagged"], lw=2, ls="--", color=COLORS[1], label="flagged in train (last < T)")
ax.plot(side.index, side["test flagged"], lw=2, color=COLORS[1], label="flagged in test (last ≥ T)")
for s in split_starts:
    ax.axvline(pd.to_datetime(s), color="grey", ls=":", lw=1)
ax.set_xlabel("cut date T")
ax.set_ylabel("flagged customers")
ax.set_ylim(0, None)
ax.grid(alpha=0.3)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

side["test positive rate"] = side["test flagged"] / (side["test flagged"] + side["test normal"])
print(side.iloc[::3].to_string())